<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">VOICE AGENT: LOCAL REAL-TIME SPEECH PIPELINE</h2>

<h3 style="color:#44D62C; text-align:left;">Objectives of This Guide</h3>

This guide walks through setting up and running the AIKit voice agent — a fully local, real-time speech pipeline that chains three vLLM-hosted models together.

By following this guide, you will learn how to:

<ul>
  <li>Start the three required model servers (LLM, STT, TTS) using <code>rzr-aikit</code></li>
  <li>Launch the OpenAI Realtime-compatible WebSocket server</li>
  <li>Connect a microphone client and hold a live spoken conversation</li>
</ul>

<h3 style="color:#44D62C; text-align:left;">🖥️ 1. Start the Models</h3>

The voice agent requires three model servers running simultaneously, each on a different port.
<br>
Note: these models and gpu utilization splits were chosen for a laptop RTX5090 (24GB VRAM).

**Terminal 1 — LLM (port 8000)**

The main language model. Uses `rzr-aikit model run` which handles GPU configuration automatically.

Wait until all three servers print a ready message before continuing.

In [ ]:
rzr-aikit model run Qwen/Qwen3.5-4B --gpu-memory-utilization 0.5

**Terminal 2 — STT / Speech-to-Text (port 8001)**

Qwen3-ASR transcribes your microphone audio to text after the VAD detects you've finished speaking.

In [ ]:
rzr-aikit model run Qwen/Qwen3-ASR-0.6B --port 8001 --gpu-memory-utilization 0.12

**Terminal 3 — TTS / Text-to-Speech (port 8002)**

Qwen3-TTS converts the LLM's text output to audio. The `--omni` flag enables streaming audio output.

In [ ]:
rzr-aikit model run Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice --port 8002 --omni --gpu-memory-utilization 0.15

<h3 style="color:#44D62C; text-align:left;">⚙️ 2. Review the Config</h3>

`voice_agent_config.json` controls which models and endpoints the server connects to. The defaults match the commands above.

In [ ]:
{
    "stt_base_url": "http://127.0.0.1:8001/v1",
    "stt_model": "Qwen/Qwen3-ASR-0.6B",
    "llm_base_url": "http://127.0.0.1:8000/v1",
    "llm_model": "Qwen/Qwen3.5-4B",
    "llm_system_prompt": "You are a helpful voice assistant. Keep answers short and conversational. Your responses will be read aloud by a text-to-speech system, so write in plain spoken prose only — no markdown, no bullet points, no headers, no asterisks, no numbered lists, no code blocks, no special characters. If the user asks you to speak in a specific way (e.g. 'whisper', 'excited', 'calm'), begin your response with exactly one tone instruction in square brackets, e.g. [whisper], [excited], [calm], then the response text. Otherwise, output only the response text with no bracketed markers at all.",
    "tts_base_url": "http://127.0.0.1:8002/v1",
    "tts_model": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    "tts_voice": "aiden",
    "api_key": "vllm"
}

If you are using different models or running the server on a different machine, you will need to adjust the values based on the table below.

Key fields:

| Field | What it controls |
|---|---|
| `stt_base_url` / `stt_model` | Which endpoint and model handle transcription |
| `llm_base_url` / `llm_model` | Which endpoint and model generate responses |
| `tts_base_url` / `tts_model` | Which endpoint and model synthesize audio |
| `tts_voice` | Speaker voice used by the TTS model |
| `llm_system_prompt` | Instructions for the LLM's behavior and output format |

The `api_key` field is set to `"vllm"` — vLLM requires a non-empty key but doesn't validate it.

To edit the config, use the included editor by running: `nano voice_agent/voice_agent_config.json` within the docker container

<h3 style="color:#44D62C; text-align:left;">🚀 3. Start the Voice Agent Server</h3>

Run the following commands:

In [ ]:
cd ..

In [ ]:
python -m voice_agent.realtime_server &

<h3 style="color:#44D62C; text-align:left;">🎙️ 4. Connect with the Example Client</h3>

The example client reads from your microphone and plays audio through your speakers. By default, Docker containers do not have access to your system's microphone or speaker. For simplicity's sake, **this section will be done on the host by copying the commands in a new terminal**.

1. exit the Docker container. Make sure to not kill or remove the container so the models and server are still up
2. clone the AIKit repo with command `git clone https://github.com/razerofficial/aikit.git`
3. Go into the repository root `cd aikit`

### Creating the environment
From the root of the aikit repository, create a virtual env and install the necessary dependencies: 

`python -m venv .venv`  
`source .venv/bin/activate`

`pip install pyaudio openai[realtime] numpy`

**For WSL, additional permissions are required:**
<br>
In settings, go to **Privacy & security** -> **Microphone** -> toggle "Let desktop apps access your microphone" **On**
<br>

Then install this package:
`sudo apt install libasound2-plugins`

#### Additional Configuration

If you are using a different llm, you must specify the model name in the client in addition to editing the `voice_agent_config.json` as described earlier.  
1. Find the client script in `/aikit/voice_agent/examples/openai_client`
2. Edit the `MODEL` string on line 31
3. You can also adjust the `SERVER_URL` if it was started on a remote machine, but this is not recommended as it will unnecessarily increase latency.

### Starting the client
With the server running, run:  
`python -m voice_agent.examples.openai_client`

Speak into your mic. The client prints:
- `[speech detected]` when the VAD triggers
- `[speech ended — generating…]` when 2 seconds of silence is detected
- `[You]: <transcript>` after STT completes
- The LLM response tokens as they stream in

The agent responds in audio through your speakers as the TTS streams back.

<h3 style="color:#44D62C; text-align:left;">🖥️ 5. (Alternative) Use the Web UI for the Voice Agent</h3>

As an alternative to the script, you can use the built-in WebUI to interact with the voice agent in your browser.  
This UI can be started from this notebook in the docker container, with no additional setup needed on the host.  
Once the server is running, open **[http://localhost:7860/](http://localhost:7860/)** to access the interface.

Note: the Gradio-based UI is much heavier and will have longer warmup/latency. The script above is recommended for a true real-time voice agent.

In [ ]:
rzr-aikit ui run

<h3 style="color:#44D62C; text-align:left;">🎛️ 6. Tuning</h3>
Depending on your physical environment or hardware, you may want to adjust the parameters set in `vad.py`

In [ ]:
# Current VAD settings — for reference
VAD_THRESHOLD     = 200  # RMS amplitude (0–32767). Raise if background noise triggers false starts.
VAD_ONSET_CHUNKS  = 3    # 3 × 100ms = 300ms of sound needed to start recording
VAD_OFFSET_CHUNKS = 20   # 20 × 100ms = 2s of silence needed to stop recording

**Emotion / tone in responses**

The TTS model supports tone instructions. The LLM is prompted to include a `[tone]` tag only when the user asks for it. Examples:

- *"respond with a whisper"* → LLM outputs `[whisper] Here is your answer...`
- *"sound excited"* → `[excited] Sure, here is...`
- Normal question → plain text, no tag, default TTS voice

The system prompt in `voice_agent_config.json` controls this behavior. Edit `llm_system_prompt` to change the LLM's defaults.

**GPU memory**

The three `--gpu-memory-utilization` values sum to `0.77` on one GPU. Adjust them if you have VRAM pressure or want headroom for other tasks:

| Model | Flag | Default utilization |
|---|---|---|
| Qwen3.5-4B (LLM) | `--gpu-memory-utilization` | 0.50 |
| Qwen3-ASR-0.6B (STT) | `--gpu-memory-utilization` | 0.12 |
| Qwen3-TTS (TTS) | `--gpu-memory-utilization` | 0.15 |

<h3 style="color:#44D62C; text-align:left;">🔧 7. Troubleshooting</h3>

**No audio output / TTS silent**
- Check the TTS server started with `--omni`. Without it, streaming audio responses are not enabled.
- Verify port 8002 is up using the health check cell above.

**LLM outputs markdown (asterisks, bullet points read aloud)**
- The system prompt instructs the LLM to avoid markdown, but it may slip on technical questions.
- Add a post-processing strip in `pipeline.py` before the TTS call if this is persistent.

**VAD triggers on background noise**
- Raise `VAD_THRESHOLD` in `vad.py` (default `200`). Try `400–600` in a noisy environment.

**`ModuleNotFoundError: No module named 'voice_agent'`**
- Run from the repo root

**Connection refused on WebSocket**
- Make sure the server is running before connecting the client.
- Default address is `ws://127.0.0.1:8081/v1/realtime`.

<h3 style="color:#44D62C; text-align:left;">🛑 8. Stop the Models and UI</h3>

Stop the gradio UI and all three running models:

In [ ]:
rzr-aikit model stop

In [ ]:
rzr-aikit ui stop

---

<h3 style="color:#44D62C; text-align:left;">✅ Summary</h3>

You've successfully set up and run a fully local voice agent powered by Razer AIKit.

##### What You've Achieved

- Launched three vLLM model servers (LLM, STT, TTS) and verified they are healthy
- Inspected and customized the voice agent configuration
- Started the OpenAI Realtime-compatible WebSocket server
- Connected a microphone client for live spoken conversation
- Learned how the VAD → STT → LLM → TTS pipeline overlaps for low latency
- Tuned VAD thresholds, silence windows, and GPU memory utilization

##### Use Cases This Enables

- Fully offline voice assistants with no cloud dependency
- Custom voice personas and tone-controlled TTS responses
- Integration with any OpenAI Realtime-compatible client or SDK
- Low-latency speech interfaces for applications running on Razer hardware